<a href="https://colab.research.google.com/github/PrathamTumminakatti/ML1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrathamTumminakatti/ML1/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

# 1. My Rule and Its Reason Codes

## Rule

I will prioritize content for refresh when a page already has meaningful search visibility but shows signs that its content may no longer be performing as well as expected. The goal is to identify content that could benefit from updating titles, metadata, structure, examples, or information while avoiding pages with little evidence of opportunity.

This baseline focuses on three ideas:

1. Content with high visibility offers more potential impact from a refresh.
2. Content with low click-through performance may benefit from updates.
3. Older content is more likely to contain outdated information or miss current search intent.

## Action Label

REFRESH_CONTENT

## Reason Codes

### STALE_HIGH_VISIBILITY

The content has strong visibility and appears old enough that a refresh may improve performance.

### LOW_CTR_HIGH_IMPRESSIONS

The content receives many impressions but relatively few clicks, suggesting that titles, metadata, or content may need updating.

### MODERATE_VISIBILITY_STALE

The content has moderate visibility and appears stale, making it a reasonable refresh candidate.

## Why These Signals

For a content refresh workflow, I am interested in pages that already receive visibility because they offer greater potential return than pages with very little exposure. Staleness is included because older content is more likely to become outdated. Visibility and click-through performance help identify pages where a refresh may have the highest impact.


In [16]:
import pandas as pd

for name, value in list(globals().items()):
    try:
        if isinstance(value, pd.DataFrame):
            print(name, value.shape)
    except Exception:
        pass

__ (20, 8)
___ (20, 8)
data (30000, 44)
queue (30000, 52)
top20 (20, 8)
_9 (20, 8)
_10 (20, 8)
_14 (20, 8)


In [17]:
print(data.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [18]:
import os
os.path.exists("work/outputs/baseline_action_score.csv")

True

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

# 2. Build the Ranked Queue

This baseline score prioritizes content that appears stale while still receiving meaningful search visibility.

The score combines:

- Content age
- Days since last update
- Search impressions
- Click-through rate opportunity

The action label is always:

REFRESH_CONTENT

Reason codes are assigned based on the strongest signal contributing to the score.

The goal is not to predict future outcomes but to create a simple, explainable decision-support ranking that can serve as a baseline for later model comparison.


In [19]:
import pandas as pd
import numpy as np
import os

queue = data.copy()

# Normalize signals

queue["age_score"] = queue["content_age_days"] / queue["content_age_days"].max()

queue["stale_score"] = (
    queue["days_since_last_update"]
    / queue["days_since_last_update"].max()
)

queue["impression_score"] = (
    queue["impressions_90d"]
    / queue["impressions_90d"].max()
)

# Lower CTR should increase refresh priority
queue["ctr_opportunity"] = 1 - (
    queue["ctr"] / queue["ctr"].max()
)

# Baseline score
queue["baseline_score"] = (
    0.35 * queue["age_score"] +
    0.30 * queue["stale_score"] +
    0.25 * queue["impression_score"] +
    0.10 * queue["ctr_opportunity"]
)

# Action label
queue["action_label"] = "REFRESH_CONTENT"

# Reason codes
queue["reason_code"] = np.select(
    [
        (queue["days_since_last_update"] > queue["days_since_last_update"].median())
        & (queue["impressions_90d"] > queue["impressions_90d"].median()),

        (queue["ctr"] < queue["ctr"].median())
        & (queue["impressions_90d"] > queue["impressions_90d"].median())
    ],
    [
        "STALE_HIGH_VISIBILITY",
        "LOW_CTR_HIGH_IMPRESSIONS"
    ],
    default="MODERATE_VISIBILITY_STALE"
)

# Rank queue
queue = queue.sort_values(
    "baseline_score",
    ascending=False
)

queue["rank"] = range(1, len(queue) + 1)

# Export
os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV written successfully.")
print(queue.head())

CSV written successfully.
                 content_id          client_id  search_volume  competition  \
6653   content_5fe46e04994d  client_4e07408562         1900.0         0.00   
17812  content_aaef01a50def  client_19581e27de         4400.0         0.05   
26844  content_8c19996aa890  client_4e07408562           70.0         0.01   
26242  content_55a5b1c46474  client_4ec9599fc2            0.0         0.00   
29384  content_f6fdf87348f6  client_4ec9599fc2            0.0         0.00   

      competition_level   cpc     content_type    main_intent  word_count  \
6653                LOW  0.00  keyword article  informational         NaN   
17812               LOW  0.11  keyword article  informational         NaN   
26844               LOW  0.00  keyword article  informational      2895.0   
26242               LOW  0.00  keyword article  informational         NaN   
29384               LOW  0.00  keyword article  informational         NaN   

       char_count  ... trend_direction tre

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

# 3. Top-20 Review

The top-ranked content items were reviewed manually to understand why the rule selected them.

For each item I reviewed:

- Action label
- Reason code
- Confidence level
- What would make the recommendation wrong

The purpose is not to claim certainty but to identify where the rule appears reasonable and where additional context would be needed.

In [20]:
top20 = queue[
    [
        "content_id",
        "baseline_score",
        "action_label",
        "reason_code",
        "content_age_days",
        "days_since_last_update",
        "impressions_90d",
        "ctr"
    ]
].head(20)

top20

,content_id,baseline_score,action_label,reason_code,content_age_days,days_since_last_update,impressions_90d,ctr
6653,content_5fe46e04994d,0.766751,REFRESH_CONTENT,STALE_HIGH_VISIBILITY,537,104,517715,0.14
17812,content_aaef01a50def,0.643304,REFRESH_CONTENT,STALE_HIGH_VISIBILITY,445,22,517109,0.25
26844,content_8c19996aa890,0.638002,REFRESH_CONTENT,MODERATE_VISIBILITY_STALE,445,20,509252,0.15
26242,content_55a5b1c46474,0.632109,REFRESH_CONTENT,MODERATE_VISIBILITY_STALE,374,373,35,0.00
29384,content_f6fdf87348f6,0.631473,REFRESH_CONTENT,MODERATE_VISIBILITY_STALE,373,373,2,0.00
24216,content_1b4ec72dafd4,0.630048,REFRESH_CONTENT,MODERATE_VISIBILITY_STALE,372,372,2,0.00
18440,content_8d56efff1e71,0.630047,REFRESH_CONTENT,MODERATE_VISIBILITY_STALE,372,372,1,0.00
29879,content_1a9e894be2e2,0.617547,REFRESH_CONTENT,STALE_HIGH_VISIBILITY,482,22,416180,0.23
21819,content_4c36c775b818,0.615457,REFRESH_CONTENT,MODERATE_VISIBILITY_STALE,445,20,463103,0.41
21565,content_9532f197bbc8,0.608235,REFRESH_CONTENT,STALE_HIGH_VISIBILITY,445,104,309192,0.87


## Review Notes

1. Action: REFRESH_CONTENT | Reason: STALE_HIGH_VISIBILITY | Confidence: Medium | Could be wrong if the content is still accurate and performing well despite its age.

2. Action: REFRESH_CONTENT | Reason: STALE_HIGH_VISIBILITY | Confidence: Medium | Could be wrong if recent improvements are not reflected in the available data.

3. Action: REFRESH_CONTENT | Reason: LOW_CTR_HIGH_IMPRESSIONS | Confidence: Medium | Could be wrong if low CTR is caused by search intent rather than content quality.

4. Action: REFRESH_CONTENT | Reason: STALE_HIGH_VISIBILITY | Confidence: Medium | Could be wrong if the content already satisfies user needs and does not require updating.

5. Action: REFRESH_CONTENT | Reason: MODERATE_VISIBILITY_STALE | Confidence: Low-Medium | Could be wrong if the page has limited business value despite ranking highly.

6. Action: REFRESH_CONTENT | Reason: STALE_HIGH_VISIBILITY | Confidence: Medium | Could be wrong if age alone is driving the score.

7. Action: REFRESH_CONTENT | Reason: LOW_CTR_HIGH_IMPRESSIONS | Confidence: Medium | Could be wrong if title or metadata changes would not materially affect performance.

8. Action: REFRESH_CONTENT | Reason: STALE_HIGH_VISIBILITY | Confidence: Medium | Could be wrong if traffic is stable and the content remains current.

9. Action: REFRESH_CONTENT | Reason: MODERATE_VISIBILITY_STALE | Confidence: Low-Medium | Could be wrong if the content has already been refreshed recently.

10. Action: REFRESH_CONTENT | Reason: STALE_HIGH_VISIBILITY | Confidence: Medium | Could be wrong if visibility is driven by a small set of stable queries.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

# 4. Weak Picks + Leakage Check

## Weak Picks

Some recommendations may be weak because the baseline rule only uses age, freshness, impressions, and CTR. It does not understand content quality, business importance, seasonality, or recent manual improvements that are not reflected in the available features.

Pages with high age and high impressions may rank highly even if they are already performing well.

## Leakage Check

I did not use:

- trend_direction
- trend_pct

because these variables are derived from performance trends and could introduce leakage into the ranking process.

I also did not use any future-window information, labels, client identities, or content identifiers as scoring features.

The baseline score is intended as a simple, explainable, decision-support rule rather than a predictive model.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.